In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# add your code here - consider creating a new cell for each section of code
# merge datasets
df = pd.merge(df_ratings, df_books, on='isbn')

# remove users with less than 200 ratings
user_counts = df['user'].value_counts()
df = df[df['user'].isin(user_counts[user_counts >= 200].index)]

# remove books with less than 100 ratings
book_counts = df['title'].value_counts()
df = df[df['title'].isin(book_counts[book_counts >= 100].index)]

# create pivot table
book_pivot = df.pivot_table(index='title', columns='user', values='rating').fillna(0)

# convert to sparse matrix
book_matrix = csr_matrix(book_pivot.values)

# build KNN model
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_matrix)



In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book=""):

    if book not in book_pivot.index:
        return []

    # get index of the book
    book_index = book_pivot.index.get_loc(book)

    # find nearest neighbors
    distances, indices = model.kneighbors(
        book_pivot.iloc[book_index, :].values.reshape(1, -1),
        n_neighbors=6
    )

    # store recommendations
    recommended_books = [book, []]

    # skip first result because it is the queried book itself
    recs = []
    for i in range(1, len(distances.flatten())):
        recs.append([
            book_pivot.index[indices.flatten()[i]],
            float(distances.flatten()[i])
        ])

    # reverse order to match FCC expected output
    recs.reverse()

    recommended_books[1] = recs

    return recommended_books

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()